<a href="https://colab.research.google.com/github/niveditha-bh/WorkflowLLM/blob/main/dataset_reduced_with5%25_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# 1. Install packages (if fresh runtime)
!pip install -q transformers peft bitsandbytes accelerate datasets
!pip install -U "bitsandbytes>=0.46.1"

In [12]:
# 2. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# 3. Hugging Face login
from huggingface_hub import login
login()

In [4]:
# 4. Load tokenizer
from transformers import AutoTokenizer

MODEL_NAME = "meta-llama/Llama-3.2-1B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded.")

Tokenizer loaded.


In [15]:
# 5. Load model in 4-bit
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map="auto")
print("Model loaded.")

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Model loaded.


In [16]:
# 6. Attach LoRA
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora_config)
print("LoRA attached.")

LoRA attached.


In [17]:
# 7. Load your 5% subset
import json

with open('/content/drive/MyDrive/dataset/train_5pct.json') as f:
    subset = json.load(f)
print("Examples:", len(subset))

Examples: 5278


In [18]:
# 8. Format into JSON-output style
def format_example(ex):
    query = ex.get("query", "")
    workflow_code = ex.get("workflow_code", "")
    task_plan = ex.get("task_plan", "")
    apis = ex.get("apis", [])

    output_json = {
        "task": query,
        "apis_used": apis,
        "plan": task_plan if task_plan else "",
        "workflow_code": workflow_code
    }
    json_output_str = json.dumps(output_json, ensure_ascii=False)
    return f"### Task:\n{query}\n\n### JSON Output:\n{json_output_str}"

texts = [format_example(ex) for ex in subset]
print(texts[0][:300])

### Task:
How can I create a health and fitness tracking system that allows users to log their daily workouts, including the type of exercise, duration, and calories burned? This system should also enable users to set reminders for their workouts, search for nearby gyms or fitness classes, and provi


In [19]:
# 9. Tokenize
from datasets import Dataset

dataset = Dataset.from_dict({"text": texts})

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=1536, padding="max_length")

tokenized_dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])
print(tokenized_dataset)

Map:   0%|          | 0/5278 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 5278
})


In [20]:
# 10. Train — 5% data, 2 epochs, JSON format, fresh run
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
import os

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/dataset/workflowllm-lora-1b-5pct-json",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=25,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(model=model, args=training_args, train_dataset=tokenized_dataset, data_collator=data_collator)

last_checkpoint = None
if os.path.isdir(training_args.output_dir):
    checkpoints = [d for d in os.listdir(training_args.output_dir) if d.startswith("checkpoint")]
    if checkpoints:
        last_checkpoint = os.path.join(training_args.output_dir, sorted(checkpoints)[-1])
        print(f"Resuming from: {last_checkpoint}")

trainer.train(resume_from_checkpoint=last_checkpoint)

Resuming from: /content/drive/MyDrive/dataset/workflowllm-lora-1b-5pct-json/checkpoint-660


Step,Training Loss


TrainOutput(global_step=660, training_loss=0.0, metrics={'train_runtime': 0.0027, 'train_samples_per_second': 3910880.048, 'train_steps_per_second': 244522.625, 'total_flos': 9.48374056552366e+16, 'train_loss': 0.0, 'epoch': 2.0})

In [21]:
import json
import torch

def run_test_json(task):
    prompt = f"### Task:\n{task}\n\n### JSON Output:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=300, do_sample=True,
            temperature=0.3, top_p=0.9, repetition_penalty=1.3,
            no_repeat_ngram_size=3, pad_token_id=tokenizer.eos_token_id
        )

    input_length = inputs['input_ids'].shape[1]
    result = tokenizer.decode(output[0][input_length:], skip_special_tokens=True)

    print("--- Raw output ---")
    print(result)
    print()

    try:
        parsed = json.loads(result)
        print("✅ Valid JSON produced")
        print(json.dumps(parsed, indent=2))
        return parsed
    except json.JSONDecodeError as e:
        print("❌ Invalid JSON —", str(e))
        return None

In [22]:
run_test_json("Create a workflow that sets an alarm and then shows an alert to confirm it was created.")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


--- Raw output ---
{"task": "Create aworkflowthatsetsanalarmandthenshowsthealerttoconfirmitwascreated.", "apis_used": ["com_apple_AirDrop_OpenDocumentIntent", "is_workflow_actions_alert"], "plan': '1. **Start**: Begin the process.\n2. **Set Alarm Time**:\n   - Call `set_alarm_time()` with parameters for time, date, duration (in minutes), repeat mode (`AlarmRepeatModeNever` or `AlarmRepeatmodeHourly`).\n3. **Show Alert Confirmation Prompt**:\nThe prompt asks if you want to set this alarm; when confirmed,\nis_workflow_actions_showresult() is called using \nthe function argument as input from user choice in previous step.'"}

❌ Invalid JSON — Expecting ':' delimiter: line 1 column 610 (char 609)


In [23]:
run_test_json("Create a workflow that checks the weather and sends me a notification if it's going to rain.")

--- Raw output ---
{"task": "Create aworkflowthatcheckstheweatherandsendsmenotificationifit’sgoingtorain.", "apis_used": [], "plan": "1. **Start**: Begin execution of this process.\n2. **Check Weather Conditions**:\n   - Call `is_workflow_actions_weather_conditions()` with no parameters, storing result in variable: \"WeatherConditions\".\n3. \n4. **If Condition 1 (Rainfall)**:\n    - Check whether 'rain' is present as part of the current conditions using `match` operator; store its value into variable : \"Is Rain?\"\n5. \nThe following actions are taken based on condition\n6. If yes for Is Rain?\n7. -- Send Notification Action via Workflow Actions Applet\ndeclaring action name=\"Send Alert!\">\nsending message about rainfall alert...\n8. End the match statement;\nit will continue checking next time.\n9. Else case,\nis not raining so proceed further without sending any alerts or notifications.",
"workflow_code": "# Checks the user inputted text against specific keywords like ''raining''

In [24]:
run_test_json("Create a workflow that reads the latest emails and creates reminders for anything marked urgent.")

--- Raw output ---
{"task": "Create aworkflowthatreadsthelatestemailsandcreatesremindersforanythingmarkedurgent.", "apis_used": [], "plan": "1. **Start**: Begin execution of this process.\n2. **Initialize Variables**:\n   - Create variables `is_workflow_actions_getlastemail` to retrieve last email from iCloud, \n     assign it as 'LastEmail'.\n3. **Check if Last Email Exists? (if True)**: If yes,\n4. **Retrieve Latest Emails in Inbox Folder:** Assign result to variable called \"LatestInbox\";\n5. **Get Current Date & Time:** Retrieve current date/time using function call\n    `- is_workflow_actions_date()`; store results into new named dictionary object with key name=\"date\"\n6. **Extract Day Name From Date/Time Result:** Use extracted day names' string representation stored previously by user inputting their choice via prompt dialog box or other means such as web scraping tools like BeautifulSoup etc., then convert them back to numeric values representing days on calendar format 0-11

In [ ]:
import json

# Load your 5% subset
with open('/content/drive/MyDrive/dataset/train_5pct.json') as f:
    subset_5pct = json.load(f)

# Extract all unique APIs actually present
apis_in_5pct = set()
for item in subset_5pct:
    if 'apis' in item:
        apis_in_5pct.update(item['apis'])

print("Total unique APIs in your 5% subset:", len(apis_in_5pct))
print()

# Filter for the core, general-purpose workflow actions (most useful for simple test tasks)
core_actions_5pct = sorted([api for api in apis_in_5pct if 'is.workflow.actions' in api.lower() or 'is_workflow_actions' in api.lower()])

print("Core workflow actions in 5% subset:", len(core_actions_5pct))
for api in core_actions_5pct:
    print(" -", api)